# Explore behavioral association within each breed

This chapter searches the measured autosomal markers for associations with behavioral elimination, separately in Labrador Retrievers (LR) and German Shorthaired Pointers (GSP). These supporting discovery analyses ask a broader question than the predefined historical-target comparison in Chapter 06.

GEMMA is the laboratory reporting workflow. GMMAT provides a comparison with a model designed for binary outcomes. Both use the same counted alleles, dogs, sex adjustment and genetic relationship matrices. We report GEMMA’s Wald, likelihood-ratio and score tests separately, alongside the GMMAT score test. Scheme A supplies the primary comparison; Scheme B checks sensitivity to the accepted additional controls. The case sets are identical within each breed.

This is the measured-genotype baseline. Reference-panel imputed variants are reserved for a later decision. With seven LR cases and nine GSP cases, the scans are exploratory and require careful assessment before a discovery claim.

## The dogs and measured markers

| Breed | Scheme | Controls | Cases | Dogs | Aligned candidate markers |
| --- | --- | ---: | ---: | ---: | ---: |
| LR | A: primary | 39 | 7 | 46 | 168,341 |
| LR | B: sensitivity | 41 | 7 | 48 | 168,341 |
| GSP | A: primary | 96 | 9 | 105 | 165,064 |
| GSP | B: sensitivity | 107 | 9 | 116 | 165,064 |

These markers passed the earlier quality filters and reference alignment. We additionally require a minor allele frequency of at least 2.5% within each analysis cohort. The minor allele is the less common allele in that cohort. Counts of markers retained for each scan are reported below.

A small fraction of measurements are missing: approximately 0.22% for LR and 0.076% for GSP in Scheme A. For the numerical scan inputs, both methods use the observed cohort mean at the missing entries. Original measurements remain unchanged, and the number of actual observations is reported separately. This does not introduce reference-panel imputed variants. Chapter 06 retains its separate complete-case target specification.

In [ ]:
from pathlib import Path
import sys
study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, run_script, file_digest, study_path, verify_datasets
alignment = read_json(study_root / "data/aligned/manifest.private.json")
assert alignment["status"] == "complete"
verify_datasets(study_root, alignment)
print("Four aligned genotype datasets verified.")

## Account for genetic relationships

Both models adjust for sex and genetic similarity among dogs of the same breed. The similarity calculation uses LD-pruned background markers: a subset that reduces the repetition of strongly correlated genetic information. We retain their aligned chromosome locations and exclude the chromosome being tested from each relationship matrix.

GMMAT models the behavioral outcome as binary. GEMMA uses a linear mixed-model approximation to the same 0/1 outcome. GEMMA remains the reporting workflow based on laboratory practice; GMMAT provides a methodological comparison. Its results are not independent replication because the same dogs and genotypes are analyzed. The three GEMMA tests are also alternative assessments of the same marker, not independent evidence. The small number of cases remains a limitation even after relatedness is included.

In [ ]:
import numpy as np
from replication_models import standardized_background

def chromosome_excluded_relationship(standardized_genotypes, chromosomes, tested_chromosome, indices):
    """Average genetic similarity outside the chromosome being tested."""
    background = standardized_genotypes[chromosomes != tested_chromosome][:, indices]
    return background.T @ background / len(background)

breed = "GSP"
background_prefix = study_root / f"data/structure/20261002-pca-umap-tsne/{breed}/02_pruned_genotypes"
aligned_bim = study_root / f"data/aligned/not-behelim-vs-behelim/{breed}/2024TSA_{breed}.bim"
background_ids, chromosomes, standardized_genotypes = standardized_background(background_prefix, aligned_bim)
print(breed, "aligned background markers:", len(chromosomes))

## Run the measured baseline

The analysis script runs each chromosome separately in both breeds and both control comparisons. It uses the association tools installed for Chapter 06. A fresh scan requires the WSL/Linux kernel and the native environment installed with `bash scripts/setup_association_wsl.sh`.

The cell below reuses completed scans when available. To run a different specification, choose a new output folder and review the settings in `config/gwas.json` first. Completed results are checked before reuse.

In [ ]:
gwas_folder = study_root / "data/gwas/measured-20261004"
manifest_path = gwas_folder / "manifest.private.json"
if not manifest_path.exists():
    run_script(study_root, "breed_gwas.py", "--output", gwas_folder, linux=True)
scans = read_json(manifest_path)
assert scans["status"] == "paired_measured_scans_complete"
assert scans["chromosomes"] == list(range(1, 39))
assert len(scans["jobs"]) == 152
saved_plan = read_json(gwas_folder / "plan.private.json")
assert saved_plan["config"] == read_json(study_root / "config/gwas.json")
for name, digest in saved_plan["scripts"].items():
    assert file_digest(study_root / "scripts" / name) == digest
for source, digest in saved_plan["sources"].items():
    assert file_digest(study_path(study_root, source)) == digest
print("Completed: 38 chromosomes × two breeds × two control comparisons.")

### Check a relationship calculation

This example reconstructs the first GSP chromosome's matrix and compares it with the input supplied to both tools. Only aggregate confirmation is displayed.

In [ ]:
import csv
example = next(job for job in scans["jobs"] if job["breed"] == "GSP")
folder = study_path(study_root, example["folder"])
with (folder / "dogs.private.tsv").open(encoding="utf-8", newline="") as handle:
    dogs = list(csv.DictReader(handle, delimiter="\t"))
positions = {dog_id: i for i, dog_id in enumerate(background_ids)}
indices = [positions[dog["id"]] for dog in dogs]
reconstructed = chromosome_excluded_relationship(
    standardized_genotypes, chromosomes, example["chromosome"], indices)
assert np.allclose(reconstructed, np.loadtxt(folder / "kinship.txt"), atol=1e-12)
print("GSP relationship calculation matches the supplied matrix.")

## Review the genome-wide results

A Manhattan plot arranges markers by chromosome and genomic position. Higher points have smaller p-values. Neighboring points can represent the same correlated genetic signal; a row of high points is not a collection of independent discoveries.

We apply Bonferroni correction within each breed-specific analysis, without additional correction across breeds, methods or GEMMA test types:

| Breed | Calculation | Cutoff |
| --- | --- | ---: |
| LR | 0.05 / 168,341 candidate markers | 2.97 × 10⁻⁷ |
| GSP | 0.05 / 165,064 candidate markers | 3.03 × 10⁻⁷ |

The same breed-specific line is used in each panel and in both control comparisons. This controls the comparison within the reported scan; it does not claim a single 5% error rate across every result in the chapter. The original candidate counts are retained even when some markers are excluded by the cohort frequency filter.

GEMMA’s Wald test evaluates an estimated effect relative to its uncertainty. The likelihood-ratio test compares fits with and without a marker effect. The score test evaluates evidence from the model without that effect. They can differ substantially in these small cohorts, so the test name accompanies every result. We do not select the smallest of the three as independent corroboration. Crossing a line identifies a candidate for assessment, not automatic biological causation or validated small-sample behavior.

In [ ]:
figure_folder = study_root / "results/private/version2/07_breed_specific_gwas"
run_script(study_root, "summarize_gwas.py", "--run-dir", gwas_folder,
           "--results-dir", figure_folder)
summary = read_json(figure_folder / "summary.private.json")
reporting = read_json(study_root / "config/gwas_reporting.json")
assert summary["reporting_policy_sha256"] == file_digest(study_root / "config/gwas_reporting.json")
assert reporting["reporting_method"] == "GEMMA"
assert reporting["comparison_method"] == "GMMAT"
from IPython.display import Markdown, Image, display
lines = ["| Breed | Comparison | Eligible markers | Dogs | Cases |",
         "| --- | --- | ---: | ---: | ---: |"]
for group in summary["groups"]:
    label = "A: primary" if group["scenario"] == scans["config"]["primary_cohort"] else "B: sensitivity"
    lines.append(f"| {group['breed']} | {label} | {group['eligible_markers']:,} | {group['dogs']} | {group['cases']} |")
display(Markdown("\n".join(lines)))

In [ ]:
for breed in ["LR", "GSP"]:
    display(Markdown(f"### {breed}: primary measured scan"))
    display(Image(filename=str(figure_folder / f"{breed}_success-vs-behelim_manhattan.png")))

### What do the primary scans show?

The LR primary scan retained 167,571 measured markers and the GSP primary scan retained 162,990. GEMMA’s Wald and likelihood-ratio tests identify markers below their within-analysis cutoffs. Neither GEMMA’s score test nor the GMMAT score test crosses its cutoff.

The table below lists each test’s minimum p-value and candidate-marker count. These counts are not counts of independent regions or confirmed discoveries. The disagreement between tests is important to interpretation; the chapter should not be summarized simply as “no hits.”

## Look at the overall p-value distribution

A QQ plot compares the observed p-values with the distribution expected under an ideal null model. Widespread departures can reflect model behavior, population differences, or associations; an isolated tail may reflect a small number of regions. Related markers and sparse genotypes complicate this interpretation.

The median chi-square ratio summarizes the middle of the distribution. A value near one or a visually satisfactory QQ plot does not validate the extreme probabilities needed for a genome-wide claim. The targeted comparison already showed unstable effect fits and uncertain variance estimates in these small cohorts.

In [ ]:
for breed in ["LR", "GSP"]:
    display(Image(filename=str(figure_folder / f"{breed}_success-vs-behelim_qq.png")))
lines = ["| Breed | Method | Test | Valid tests | Smallest p | Within-analysis cutoff | Markers below cutoff | Median chi-square ratio |",
         "| --- | --- | --- | ---: | ---: | ---: | ---: | ---: |"]
for result in summary["tests"]:
    if result["scenario"] != scans["config"]["primary_cohort"]: continue
    lines.append(f"| {result['breed']} | {result['method']} | {result['test']} | {result['valid_tests']:,} | "
                 f"{result['minimum_p']:.3g} | {result['threshold']:.3g} | {result['below_analysis_threshold']} | "
                 f"{result['median_chi_square_ratio']:.3f} |")
display(Markdown("\n".join(lines)))

All 152 GMMAT chromosome null models converged, and both methods returned finite score-test p-values at every eligible marker. The median chi-square ratios were 1.140 / 1.105 for LR and 1.083 / 1.074 for GSP (GMMAT / GEMMA). The plots show departures from the ideal null distribution, especially in the tails; these ratios do not establish reliable genome-wide probabilities.

GEMMA's alternative-model variance ratio reached a search boundary at every LR marker and almost every GSP marker. Some LR alternative fits also had unavailable effect estimates. These limitations affect effect interpretation and remain visible in the private diagnostics.

## Check sensitivity to the control definition

Scheme B adds the accepted nonbehavioral controls while retaining the same cases. Its Manhattan and QQ plots show how results change with that definition. The two schemes share dogs and therefore are not independent studies. We retain Scheme A as primary even if Scheme B produces a smaller p-value.

In [ ]:
for breed in ["LR", "GSP"]:
    display(Markdown(f"### {breed}: sensitivity comparison"))
    display(Image(filename=str(figure_folder / f"{breed}_not-behelim-vs-behelim_manhattan.png")))
    display(Image(filename=str(figure_folder / f"{breed}_not-behelim-vs-behelim_qq.png")))

Scheme B retained 168,341 LR markers and 165,064 GSP markers. Its candidate counts are reported separately below using the same breed-specific cutoffs. It remains a sensitivity analysis sharing the primary cases; a smaller p-value in Scheme B does not turn it into an independent discovery cohort.

In [ ]:
lines = ["| Breed | Method | Test | Smallest p | Markers below within-analysis cutoff |",
         "| --- | --- | --- | ---: | ---: |"]
for result in summary["tests"]:
    if result["scenario"] != scans["config"]["sensitivity_cohort"]: continue
    lines.append(f"| {result['breed']} | {result['method']} | {result['test']} | "
                 f"{result['minimum_p']:.3g} | {result['below_analysis_threshold']} |")
display(Markdown("\n".join(lines)))

## Interpret leading markers with their observed counts

The following table shows the leading marker from each of GEMMA’s three tests within each breed, with all three GEMMA p-values and the GMMAT score p-value for that same marker. Repeated markers across rows are the same finding assessed by different tests. They are not separate discoveries.

Each marker’s observed allele counts, missing measurements and regional neighbors matter to interpretation. GEMMA effect estimates are on the outcome scale, not odds ratios. GMMAT score tests do not provide an effect estimate; fitting an alternative logistic model is a separate follow-up step. Sparse genotype groups and boundary variance estimates can make effect interpretation unstable.

In [ ]:
lines = ["| Breed | Leading GEMMA test | Marker | CanFam4 position | Counted allele | Case ALT / observed copies | Control ALT / observed copies | Wald p | Likelihood-ratio p | GEMMA score p | GMMAT score p |",
         "| --- | --- | --- | --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |"]
for result in summary["tests"]:
    if result["method"] != "GEMMA" or result["scenario"] != scans["config"]["primary_cohort"]: continue
    lead = result["top_variants"][0]
    lines.append(f"| {result['breed']} | {result['test']} | {lead['marker']} | chr{lead['chromosome']}:{lead['position']} | "
                 f"{lead['ALT']} | {lead['case_alt_copies']} / {2*int(lead['case_called'])} | "
                 f"{lead['control_alt_copies']} / {2*int(lead['control_called'])} | "
                 f"{lead['GEMMA_wald_p']:.3g} | {lead['GEMMA_lrt_p']:.3g} | {lead['GEMMA_p']:.3g} | {lead['GMMAT_p']:.3g} |")
display(Markdown("\n".join(lines)))
print("Converged GMMAT chromosome null models:", summary["gmmat_null_models_converged"],
      "of", summary["chromosome_null_models"])

## Scope of the conclusions

This chapter reports GEMMA as the laboratory workflow, including all three tests explicitly, and GMMAT as a methodological comparison. GEMMA’s Wald and likelihood-ratio results include candidates below their within-analysis cutoffs; the score-test results do not. These are distinct statistical results on the same dogs, not independent corroboration. Neither small case counts nor uncertain relationship effects are resolved simply by obtaining a p-value. Candidate signals require additional small-sample assessment and appropriate effect estimation before being carried forward as discoveries. Nonsignificant results do not establish absence of a genetic association.

Imputed variants have not entered these scans. Their selection and validation will be considered separately, using the marker-level evidence from Chapter 05. The separate-breed results also do not replace the independent Labrador historical-target question.

Implementation and software details are in the [breed-specific methods note](../docs/methods/07%20Breed-specific%20measured%20GWAS.md).